# OSC Pick-and-Place Training (Colab)

Phase 3, built directly on `5-arm_project_osc`'s Phase 2 (Grasp) success. Runs `train_osc_pick_place_bc_parallel.py` from the `Arm-OSC-Pick-and-Place` repo on a Colab runtime — a behavior-cloning-warm-started SAC training, applied from the start rather than trying pure RL first (see `IMP_NOTES.md`: pure RL never found a single grasp success in the grasp-only task across 585k/1M steps, and this task is a longer, harder multi-stage sequence). Cell 5.5 collects real successful pick-and-place demonstrations via a hand-scripted routine first; cell 6 seeds the replay buffer and pretrains the actor from them before RL fine-tuning begins.

**Runtime type**: a plain CPU runtime is fine — no need to select a GPU. This workload is CPU-bound MuJoCo physics plus a tiny MLP; `device="cpu"` is set explicitly in the training script regardless, and a T4 wouldn't speed this up.

**Why Drive is mounted**: Colab's local disk is wiped whenever the runtime disconnects or recycles (idle timeout, 12h session cap, etc.) — a multi-hour training run WILL eventually hit this. Checkpoints are symlinked into Google Drive below so they survive a disconnect; only re-run cells 1-4 to resume watching a run, and cells 5.5-6 again to re-collect demonstrations and continue/restart training.

In [ ]:
# Cell 1 — mount Google Drive (checkpoints and the final model save both
# land here, not on Colab's ephemeral local disk)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 — clone the repo. Leave the token prompt blank if the repo is public;
# paste a GitHub Personal Access Token (repo scope) if it's private.
import getpass, os

REPO_URL = "https://github.com/kaustubhadhe1206/Arm-OSC-Pick-and-Place.git"
REPO_DIR = "Arm-OSC-Pick-and-Place"

token = getpass.getpass("GitHub token (leave blank if repo is public): ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {clone_url}

%cd {REPO_DIR}

In [ ]:
# Cell 3 — point checkpoints at Drive via a symlink, so the training script's
# existing `save_path="./checkpoints/"` transparently writes to Drive instead
# of Colab's local (ephemeral) disk, with no changes needed to the script
# itself.
import os

DRIVE_CKPT_DIR = "/content/drive/MyDrive/Arm-OSC-Pick-and-Place-checkpoints"
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

if os.path.islink("checkpoints") or os.path.isdir("checkpoints"):
    !rm -rf checkpoints
!ln -s {DRIVE_CKPT_DIR} checkpoints

print("Checkpoints will be saved to:", DRIVE_CKPT_DIR)

In [ ]:
# Cell 4 — install dependencies. torch is already preinstalled on Colab (CUDA
# build) — that's fine, the training script forces device="cpu" regardless.
!pip install -q mujoco gymnasium stable-baselines3

In [ ]:
# Cell 5 — sanity check: how many CPU cores does this runtime actually have?
# (5-arm_project_osc's Colab sessions showed 2 vCPUs on the free tier.)
import os
print("CPU count:", os.cpu_count())

In [ ]:
# Cell 5.5 — collect pick-and-place demonstrations via a hand-scripted (no
# RL) routine, used to warm-start training below. This task's scripted
# routine succeeds ~75-85% of the time (see IMP_NOTES.md) -- somewhat
# slower per-attempt than the grasp-only task's demo collection since each
# successful episode is a full approach-grasp-lift-carry-release-settle
# sequence (~250-350 transitions vs ~150-200 for grasp-only). Only needs to
# run once per Colab session; demonstrations.npz persists in this session's
# local disk for the rest of it (re-run if the runtime disconnects and you
# start a fresh session).
!python collect_demonstrations.py 300

In [ ]:
# Cell 6 — run BC-warm-started training. This streams SB3's logging table
# live and blocks until 1,000,000 steps complete or the runtime
# disconnects — checkpoints every 12,500 steps land in Drive via the
# symlink either way, so a disconnect loses at most that much progress.
# NOTE: this task is longer/harder than the grasp-only one that needed the
# full 1M steps to fully converge -- watch ep_len_mean/ep_rew_mean near the
# end and consider extending total_timesteps in the script if it's still
# clearly improving rather than assuming 1M is automatically enough.
!python train_osc_pick_place_bc_parallel.py

In [ ]:
# Cell 7 — only relevant if cell 6 finished without disconnecting: the FINAL
# model.save() writes to the repo directory (Colab's local disk), not
# checkpoints/ — copy it to Drive too so it isn't lost.
!cp sac_franka_osc_pick_place_bc_parallel.zip /content/drive/MyDrive/Arm-OSC-Pick-and-Place-checkpoints/ 2>/dev/null || echo "Not found yet — training may not have completed."